# Notebook 28 — ERA5 Moisture + Low-Level Winds Download (MJO moisture-constraint experiment)
**Project:** ENSO-BSISO SSL — MJO extension · moisture-constraint experiment (Zhang et al. 2020 motivated)
**Author:** Jiayi (jh9141@nyu.edu)

Downloads the fields needed to diagnose the **moisture-convection phase relationship** of the MJO
(Zhang et al. 2020, *Four Theories of the MJO*). Same domain/grid/period as the existing MJO pipeline
(nb12), so everything aligns with `X_MJO` and the RMM labels.

| Field | ERA5 variable | Why |
|---|---|---|
| **TCWV** (total column water vapour) | `total_column_water_vapour` (single-level) | column moisture -> **moisture-mode** test (Pr proportional to column q) |
| **q** at 1000/925/850/700 hPa | `specific_humidity` (pressure-levels) | **lower-tropospheric** moisture (1000-700 integral) -> **skeleton** test; also gives dq/dt |
| **u,v** at 1000/925 hPa | `u/v_component_of_wind` (pressure-levels) | low-level **divergence** -> BL-convergence lead (**trio-interaction**) |

(The Rossby-Kelvin ratio needs no new data — u850 is already channel 0 of `X_MJO`.)

**Domain:** 15S-15N, all longitudes, 2x2 deg. **Period:** 1979-2023, daily mean (4x/day -> mean).
**Output (Google Drive -> `BSISO_SSL_Project/MJO/moisture_constraints/data/raw/`):**
```
TCWV_{yr}.nc          qplev_{yr}.nc          uvplev_low_{yr}.nc      (45 annual files each)
```
Needs the same CDS API key as the rest of the project.

---

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
MOIST_RAW   = f'{MJO_DIR}/moisture_constraints/data/raw'
os.makedirs(MOIST_RAW, exist_ok=True)
print('Raw moisture dir:', MOIST_RAW)
for f in sorted(os.listdir(MOIST_RAW)):
    print('  ', f, round(os.path.getsize(f'{MOIST_RAW}/{f}')/1e6, 1), 'MB')

## Cell 2 — Install CDS API Client

In [ ]:
get_ipython().system('pip install cdsapi --quiet')
import cdsapi
print('cdsapi ready.')

## Cell 3 — CDS API Credentials
Same personal access token as the BSISO/MJO project.

In [ ]:
# ============================================================
# FILL IN YOUR PERSONAL ACCESS TOKEN HERE
# https://cds.climate.copernicus.eu/  ->  Your profile
# ============================================================
CDS_API_KEY = 'YOUR_CDS_API_KEY_HERE'
# ============================================================
import os
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f'url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n')
print('CDS credentials saved.')
try:
    import cdsapi; cdsapi.Client(quiet=True); print('CDS API connection: OK')
except Exception as e:
    print('CDS API connection FAILED:', e)

## Cell 4 — Shared config + daily-mean helper

All three fields are **instantaneous** -> download 4x/day (00/06/12/18 UTC) and average to a daily mean,
exactly as nb12 did for the winds. The `pressure_level` dimension is preserved for q and u/v.

In [ ]:
import cdsapi, os
import xarray as xr

if not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    raise RuntimeError('Run Cell 3 first to set up CDS credentials.')
client = cdsapi.Client()

ALL_MONTHS = [f'{m:02d}' for m in range(1, 13)]
DAYS       = [f'{d:02d}' for d in range(1, 32)]      # CDS ignores invalid dates
INST_TIMES = ['00:00', '06:00', '12:00', '18:00']    # instantaneous -> daily MEAN
AREA       = [15, -180, -15, 180]                    # N, W, S, E  (global 15S-15N)
GRID       = [2.0, 2.0]
YEARS      = list(range(1979, 2024))

def aggregate_daily_mean(sub_file, out_file, keep_vars):
    # Collapse a 4x/day ERA5 file to a daily mean; keep pressure_level if present.
    ds = xr.open_dataset(sub_file)
    tdim = 'valid_time' if 'valid_time' in ds.dims else 'time'
    ds = ds[keep_vars].reset_coords(drop=True)
    daily = ds.resample(**{tdim: '1D'}).mean().dropna(dim=tdim, how='all')
    daily.to_netcdf(out_file)
    ds.close(); os.remove(sub_file)

print('Config ready. Years', YEARS[0], '-', YEARS[-1], ' area', AREA, ' grid', GRID)

## Cell 5 — Download TCWV (column moisture)
Single-level `total_column_water_vapour`. ~365x4 fields/yr.

In [ ]:
print(f'{len(YEARS)} annual TCWV chunks. Existing files skipped.')
for yr in YEARS:
    out = f'{MOIST_RAW}/TCWV_{yr}.nc'
    if os.path.exists(out):
        print(f'[SKIP] TCWV_{yr}.nc ({os.path.getsize(out)/1e6:.1f} MB)'); continue
    sub = out.replace('.nc', '_sub.nc')
    print(f'Downloading TCWV {yr} ...', end=' ', flush=True)
    client.retrieve('reanalysis-era5-single-levels', {
        'product_type': 'reanalysis',
        'variable'    : 'total_column_water_vapour',
        'year'        : str(yr), 'month': ALL_MONTHS, 'day': DAYS, 'time': INST_TIMES,
        'area'        : AREA, 'grid': GRID, 'data_format': 'netcdf',
    }, sub)
    aggregate_daily_mean(sub, out, keep_vars=['tcwv'])
    print(f'done ({os.path.getsize(out)/1e6:.1f} MB)')
print('\nTCWV done.')

## Cell 6 — Download specific humidity q at 1000/925/850/700 hPa
For the **lower-tropospheric** (1000-700 hPa) moisture integral, computed in nb29.

In [ ]:
Q_LEVELS = ['1000', '925', '850', '700']
print(f'{len(YEARS)} annual q chunks at levels {Q_LEVELS}. Existing files skipped.')
for yr in YEARS:
    out = f'{MOIST_RAW}/qplev_{yr}.nc'
    if os.path.exists(out):
        print(f'[SKIP] qplev_{yr}.nc ({os.path.getsize(out)/1e6:.1f} MB)'); continue
    sub = out.replace('.nc', '_sub.nc')
    print(f'Downloading q {yr} ...', end=' ', flush=True)
    client.retrieve('reanalysis-era5-pressure-levels', {
        'product_type'  : 'reanalysis',
        'variable'      : 'specific_humidity',
        'pressure_level': Q_LEVELS,
        'year'          : str(yr), 'month': ALL_MONTHS, 'day': DAYS, 'time': INST_TIMES,
        'area'          : AREA, 'grid': GRID, 'data_format': 'netcdf',
    }, sub)
    aggregate_daily_mean(sub, out, keep_vars=['q'])
    print(f'done ({os.path.getsize(out)/1e6:.1f} MB)')
print('\nq done.')

## Cell 7 — Download low-level winds u,v at 1000/925 hPa
For low-level **divergence** (BL convergence lead, trio-interaction).

In [ ]:
UV_LEVELS = ['1000', '925']
print(f'{len(YEARS)} annual u,v chunks at levels {UV_LEVELS}. Existing files skipped.')
for yr in YEARS:
    out = f'{MOIST_RAW}/uvplev_low_{yr}.nc'
    if os.path.exists(out):
        print(f'[SKIP] uvplev_low_{yr}.nc ({os.path.getsize(out)/1e6:.1f} MB)'); continue
    sub = out.replace('.nc', '_sub.nc')
    print(f'Downloading u,v {yr} ...', end=' ', flush=True)
    client.retrieve('reanalysis-era5-pressure-levels', {
        'product_type'  : 'reanalysis',
        'variable'      : ['u_component_of_wind', 'v_component_of_wind'],
        'pressure_level': UV_LEVELS,
        'year'          : str(yr), 'month': ALL_MONTHS, 'day': DAYS, 'time': INST_TIMES,
        'area'          : AREA, 'grid': GRID, 'data_format': 'netcdf',
    }, sub)
    aggregate_daily_mean(sub, out, keep_vars=['u', 'v'])
    print(f'done ({os.path.getsize(out)/1e6:.1f} MB)')
print('\nu,v done.')

## Cell 8 — Verify downloads

In [ ]:
import os, xarray as xr, numpy as np, pandas as pd
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
MOIST_RAW = '/content/drive/MyDrive/BSISO_SSL_Project/MJO/moisture_constraints/data/raw'

def check(prefix, var, levels=False):
    fs = sorted(f'{MOIST_RAW}/{f}' for f in os.listdir(MOIST_RAW)
                if f.startswith(prefix) and f.endswith('.nc'))
    print(f'\n[{prefix}] {len(fs)}/45 annual files')
    if not fs: return
    ds = xr.concat([xr.open_dataset(f) for f in fs], dim='valid_time').sortby('valid_time')
    t = pd.DatetimeIndex(ds.valid_time.values)
    print(f'  days {len(t)}  {t[0].date()}..{t[-1].date()}  months {sorted(set(t.month))}')
    print(f'  grid {len(ds.latitude)} lat x {len(ds.longitude)} lon', end='')
    if levels: print(f'  levels {sorted(ds.pressure_level.values.tolist())}')
    else: print()
    v = ds[var].values
    print(f'  {var} range [{np.nanmin(v):.3g}, {np.nanmax(v):.3g}]  NaN {int(np.isnan(v).sum())}')
    yrs = sorted(set(t.year)); miss = [y for y in range(1979,2024) if y not in yrs]
    if miss: print(f'  MISSING years: {miss}')
    ds.close()

check('TCWV_', 'tcwv')
check('qplev_', 'q', levels=True)
check('uvplev_low_', 'u', levels=True)
print('\nVerification complete. Next: nb29 (preprocess).')

---
## Done!
`MJO/moisture_constraints/data/raw/` now holds `TCWV_*.nc`, `qplev_*.nc`, `uvplev_low_*.nc` (45 each).

**Next:** `29_mjo_moisture_preprocess.ipynb` — meridional average + nb13 anomaly pipeline -> processed
`q_col`, `q_low`, low-level divergence aligned to `X_MJO`/RMM.

---
*DDCS Project | jh9141@nyu.edu*